In [29]:
from tensorflow import keras
from tensorflow.keras import layers
import re

In [30]:
# 1. Загрузка данних
max_features = 20000
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=max_features)

In [31]:
#Розбиваємо всі ревью на окремі речення за крапкою, знаком оклику чи питання.
def split_text_into_sentences(text):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    return [s.strip() for s in sentences if s.strip()]


In [32]:
#Встановлюємо значення довжини речення та створюємо обрізку та падинг для речень які занадто довгі, або коротк
maxlen_sentence = 40
x_train = keras.utils.pad_sequences(x_train, maxlen=maxlen_sentence, padding="post", truncating="post")
x_test = keras.utils.pad_sequences(x_test, maxlen=maxlen_sentence, padding="post", truncating="post")

In [33]:
#Створення моделі Bidirectional LSTM для коротких послідовностей, ембедінгу та класифікації
inputs = layers.Input(shape=(None,), dtype="int32")
x = layers.Embedding(max_features, 128)(inputs)
x = layers.Bidirectional(layers.LSTM(32, return_sequences=True))(x)
x = layers.Bidirectional(layers.LSTM(16))(x)
x = layers.Dense(64, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_2 (Embedding)         │ (None, None, 128)      │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, None, 64)       │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 32)             │        10,368 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         2,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,613,761 (9.97 MB)

 Trainable params: 2,613,761 (9.97 MB)

 Non-trainable params: 0 (0.00 B)

In [34]:
# Компіляція моделі
model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

In [35]:
# Навчання моделі
batch_size = 32
epochs = 3

history = model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_data=(x_test, y_test),
    verbose=2
)

Epoch 1/3
782/782 - 17s - 21ms/step - accuracy: 0.7176 - loss: 0.5380 - val_accuracy: 0.7375 - val_loss: 0.5237
Epoch 2/3
782/782 - 18s - 23ms/step - accuracy: 0.8445 - loss: 0.3590 - val_accuracy: 0.7437 - val_loss: 0.5396
Epoch 3/3
782/782 - 15s - 19ms/step - accuracy: 0.9178 - loss: 0.2097 - val_accuracy: 0.7302 - val_loss: 0.7520
